# DnD Implementation

Let's get some enums and stuff out of the way.

## Races

In [ ]:
for i in range(0, 1):
    print(i)

In [25]:
from random import randint
from typing import Dict, List, Literal
from pydantic import BaseModel, Field

After handling imports, let's get an abilities generated.

In [21]:
def roll_dice(num_rolls: int = 4, die_sides: int = 6) -> int:
    # Collect a list of rolls
    rolls: List[int] = []

    # Roll a dice num_rolls times
    for roll in range(0, num_rolls):
        # What type? D6 is the default
        rand_roll = randint(1, die_sides)
        # Add roll to list
        rolls.append(rand_roll)
    
    # Sort the rolls least -> most
    rolls.sort()
    # Remove the lowest roll
    rolls.pop(0)
    # Return the sum of the remaining rolls
    return sum(rolls)

In [22]:
class Abilities(BaseModel):
    strength: int = Field(ge=0, le=30, default=roll_dice())
    dexterity: int = Field(ge=0, le=30, default=roll_dice())
    constitution: int = Field(ge=0, le=30, default=roll_dice())
    intelligence: int = Field(ge=0, le=30, default=roll_dice())
    wisdom: int = Field(ge=0, le=30, default=roll_dice())
    charisma: int = Field(ge=0, le=30, default=roll_dice())
    
    available_points: int = Field(ge=0, le=30, default=roll_dice())

    def get_modifier(self, stat: int) -> int:
        return (stat - 10) // 2

Let's get some base classes made for races and classes.

In [23]:
class BaseRace(BaseModel):
    size: Literal["Small", "Medium"] = "Small"
    speed: int = Field(ge=1, le=100, default=30)
    languages: List[str] = ["Common"]
    abilities: Abilities = Field(default_factory=Abilities)

## Default Races

Let's setup our default races.

In [ ]:
class Dragonborn(BaseRace):
    size: Literal["Medium"] = "Medium"
    speed: int = Field(ge=1, le=100, default=30)
    draconic_ancestry: Literal["Black", "Blue", "Brass", "Bronze", "Copper", "Gold", "Green", "Red", "Silver", "White"] = "Black"
    damage_type: Literal["Acid", "Lightning", "Fire", "Poison", "Cold"] = "Acid"
    breath_weapon: Literal["5x30 line", "15 cone"] = "5x30 line"
    
    def modifiers(self) -> None:
        self.abilities.strength += 2
        self.abilities.charisma += 1

        ancestry_damage: Dict[Literal["Black", "Blue", "Brass", "Bronze", "Copper", "Gold", "Green", "Red", "Silver", "White"], Literal["Acid", "Lightning", "Fire", "Poison", "Cold"]] = {
            "Black": "Acid",
            "Blue": "Lightning",
            "Brass": "Fire",
            "Bronze": "Lightning",
            "Copper": "Acid",
            "Gold": "Fire",
            "Green": "Poison",
            "Red": "Fire",
            "Silver": "Cold",
            "White": "Cold",
        }

        ancestry_weapon: Dict[Literal["Black", "Blue", "Brass", "Bronze", "Copper", "Gold", "Green", "Red", "Silver", "White"], Literal["5x30 line", "15 cone"]] = {
            "Black": "5x30 line",
            "Blue": "5x30 line",
            "Brass": "5x30 line",
            "Bronze": "5x30 line",
            "Copper": "5x30 line",
            "Gold": "15 cone",
            "Green": "15 cone",
            "Red": "15 cone",
            "Silver": "15 cone",
            "White": "15 cone",
        }

        self.damage_type = ancestry_damage[self.draconic_ancestry]
        self.breath_weapon = ancestry_weapon[self.draconic_ancestry]

## Classes

In [ ]:
class BaseClass(BaseModel):
    level: int = Field(ge=1, le=20, default=1)
    inspiration: bool = False
    proficiency_bonus: int = Field(ge=2, le=6, default=2)
    features: List[str] = []